# LLM-as-a-Judge Calibration and Evaluation

This notebook loads a 10–15 example calibration set from `calibration_set.json`, runs an LLM judge (DeepSeek or fallback) in Colab/Kaggle, stores judge outputs, and computes validation metrics (Cohen's κ, Win Rate, position-bias stats).

The notebook implements the workflow described in `docs/reports/comment_quality_evaluation.md` (Section 6).

## 1. Setup & Dependencies

Install required packages for model loading, inference, and metrics computation.

In [1]:
!pip install -q -U bitsandbytes accelerate transformers tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.3 MB/s eta 0:00:0000:01


## 2. Import Libraries

Core dependencies: PyTorch, Transformers, Pandas, scikit-learn for metrics.

In [2]:
import json, os, re
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score

## 3. Data Loading & Validation

Load calibration set from JSON. Validates schema and handles missing files gracefully.

**Schema Expected:**
- `file_id`: Unique file identifier
- `path`: File path in PR
- `patched_content`: Full file content after changes
- `human_comments`: List of human review comments
- `ai_comments`: List of AI-generated comments
- `human_score`: Ground truth score (1-5) assigned by human annotator
- `human_side`: Which position human review occupies ("A" or "B" for randomization)

In [3]:
DATASET_PATH = '/kaggle/input/datasets/se1nastol/ai-code-reviewer-calibration-set/calibration_set.json'

try:
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        calibration_set = json.load(f)
    print(f"Loaded {len(calibration_set)} examples from calibration_set.json")
except FileNotFoundError:
    print("WARNING: calibration_set.json not found. Please upload it.")
    calibration_set = []

Loaded 15 examples from calibration_set.json


## 4. Model Loading & Judge Initialization

Loads the DeepSeek-R1-Distill-Qwen-32B model for inference.

**Model Choice Rationale** (see `docs/reports/comment_quality_evaluation.md` Section 6.1.2):
- On-premise compatibility (no external APIs)
- Code-specific reasoning via chain-of-thought
- Low position bias compared to base models

In [4]:
model_id = "unsloth/DeepSeek-R1-Distill-Qwen-32B-bnb-4bit"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading 32B model across 2x T4 GPUs (this takes ~3 mins)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)
print("Model loaded successfully!")

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

Model loaded successfully!


## 5. Prompt Engineering & Judge Inference

Defines the evaluation prompt (builton Section 6.3 of methodology document) and implements JSON extraction from model outputs.

**Key Features:**
- Holistic N:M review comparison (not comment-by-comment matching)
- Anti-bias instructions explicit in system prompt
- Chain-of-thought reasoning required
- Scoring rubric (1-5) with semantic vs. stylistic distinction
- Graceful JSON parsing with fallback to neutral Tie verdict

In [ ]:
def build_prompt(entry, review_a, review_b):
    prompt = f"""You are a strict Senior Staff Software Engineer acting as a code review judge for a Python codebase.
Your task is to objectively evaluate two sets of review comments—Review A and Review B—on a single Python file.

### ARCHITECTURAL CONSTRAINTS (CRITICAL)
This system is designed ONLY to catch **merge-blocking semantic issues** (e.g., logical correctness, security vulnerabilities, thread-safety, performance regressions, or severe maintainability flaws).
**Style, formatting, and minor naming conventions are OUT OF SCOPE** (they are handled by CI linters). Comments focusing purely on style should be treated as "Noise".

### SOURCE MATERIAL

FILE PATH: {entry.get('path')}
FILE CONTENT:
```python
{(entry.get('patched_content') or '')[:2000]}
```

### REVIEWS TO EVALUATE

REVIEW A:
{json.dumps(review_a, ensure_ascii=False, indent=2)}

REVIEW B:
{json.dumps(review_b, ensure_ascii=False, indent=2)}

### EVALUATION PROTOCOL (STRICT)
You MUST evaluate both reviews using the following structured rubric.

| Score | Description |
|-------|-------------|
| 5 | **Excellent:** Review correctly identifies critical/blocking semantic issues, provides clear and actionable fixes, and avoids noise or hallucinations. No significant blocking issues were missed. |
| 4 | **Strong:** Review addresses most important blocking issues and is mostly actionable, but may miss a minor logical point or include a slight nitpick/style comment. |
| 3 | **Adequate:** Review catches some relevant issues but misses at least one critical blocking point, OR it dilutes good advice with significant stylistic noise/unnecessary comments. Value is mixed. |
| 2 | **Weak:** Review misses multiple critical blocking issues, OR is predominantly composed of incorrect, irrelevant stylistic noise. May cause developer frustration. |
| 1 | **Harmful:** Review is misleading, hallucinates code that does not exist, suggests breaking changes, or completely fails to identify obvious critical bugs. |

### EVALUATION RULES:
1. **Hallucination Check (Score 1):** If a review mentions variables, loops, or logic that DO NOT EXIST in the provided FILE CONTENT, you MUST score it a 1 (Harmful). Do not assume code exists outside the snippet.
2. **The "Noise" Penalty (Score 2 or 3):** If a review ignores critical bugs to focus purely on PEP8 formatting, docstrings, or minor naming ("refactor for clarity"), it is providing Noise. Score it a 2 or 3 depending on severity.
3. **The "Silence" Evaluation:** If a review is empty (no comments):
   - If the FILE CONTENT contains a blocking semantic bug, the empty review missed it. Score = 1 or 2.
   - If the FILE CONTENT is free of blocking semantic bugs (even if style is bad), an empty review is correct. Score = 5.
4. **Outcome Selection:**
   - If Score A > Score B: "A Win"
   - If Score B > Score A: "B Win"
   - If Score A == Score B: "Tie"

### OUTPUT FORMAT
You must output your reasoning first in a <think> block, explicitly verifying if the issues mentioned are semantic (blocking) vs stylistic (noise), and if the code actually exists.
Then, output ONLY a valid JSON object in this exact format:
```json
{{
  "review_a_score": 0,
  "review_a_reasoning": "...",
  "review_b_score": 0,
  "review_b_reasoning": "...",
  "outcome": "A Win/B Win/Tie"
}}
```"""
    return prompt


def extract_json_from_deepseek(text):
    json_match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        json_str = json_match.group(0) if json_match else "{}"

    try:
        return json.loads(json_str)
    except Exception as e:
        print(f"JSON Parse Error: {e}")
        return {'review_a_score': 3, 'review_b_score': 3, 'outcome': 'Tie', 'error': 'Parse Failed'}


def judge_infer(entry, review_a, review_b):
    prompt = build_prompt(entry, review_a, review_b)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    result = extract_json_from_deepseek(response)
    result['raw_deepseek_output'] = response
    return result

## 6. Running Bidirectional Evaluation

Implements position-bias mitigation via bidirectional scoring (Section 6.4.2 of methodology):
- **Pass 1:** Review A (Human) in Position 1, Review B (AI) in Position 2
- **Pass 2:** Positions swapped—Review B (AI) in Position 1, Review A (Human) in Position 2

Outputs are saved to `raw_judge_outputs.json` for per-file debugging.

In [ ]:
from datetime import datetime

raw_outputs = []
print("Starting evaluation (Double-blind swapping protocol)...")

for entry in tqdm(calibration_set[:5]):
    file_id = entry.get('file_id', 'unknown')

    if entry.get('human_side', 'A') == 'A':
        human_rev = entry.get('human_comments', [])
        ai_rev = entry.get('ai_comments', [])
    else:
        human_rev = entry.get('ai_comments', [])
        ai_rev = entry.get('human_comments', [])

    # Pass 1: [A=Human, B=AI]
    out1 = judge_infer(entry, human_rev, ai_rev)
    # Pass 2: [A=AI, B=Human]
    out2 = judge_infer(entry, ai_rev, human_rev)

    raw_outputs.append({
        'file_id': file_id,
        'pass1_human_vs_ai': out1,
        'pass2_ai_vs_human': out2,
        'ts': datetime.utcnow().isoformat()
    })

with open('raw_judge_outputs.json', 'w', encoding='utf-8') as f:
    json.dump(raw_outputs, f, indent=2, ensure_ascii=False)

print(f"Saved raw_judge_outputs.json for {len(raw_outputs)} files.")

Starting evaluation (Double-blind swapping protocol)...


  0%|          | 0/5 [00:00<?, ?it/s]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipykernel_55/2174982990.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'ts': datetime.utcnow().isoformat()
 20%|██        | 1/5 [07:20<29:23, 440.98s/it]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to t

JSON Parse Error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


 60%|██████    | 3/5 [22:09<14:52, 446.37s/it]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


JSON Parse Error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 80%|████████  | 4/5 [29:35<07:26, 446.33s/it]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 5/5 [37:21<00:00, 448.21s/it]

Saved raw_judge_outputs.json for 5 files.


## 7. Metrics Computation & Calibration Analysis

Computes Cohen's κ (inter-rater reliability) between manual human scores and judge predictions.

**Acceptance Criterion** (Section 6.5 of methodology):
- κ ≥ 0.70 → "Substantial Agreement" → Proceed to full 100-PR evaluation
- κ < 0.70 → Refine prompt and recalibrate

Outputs saved to:
- `metrics.json`: Summary metrics (Cohen's κ, position flips, win rate)
- `raw_judge_outputs.json`: Detailed per-file judge outputs for debugging

In [11]:
from collections import Counter

human_scores = []
judge_scores = []
final_outcomes = []
position_flips = 0

for ro in raw_outputs:
    file_id = ro['file_id']
    entry = next((e for e in calibration_set if e.get('file_id') == file_id), None)

    human_score = entry.get('human_score') if entry else None

    p1 = ro.get('pass1_human_vs_ai', {})
    p2 = ro.get('pass2_ai_vs_human', {})

    # Pass 1: A=Human, B=AI. Pass 2: A=AI, B=Human
    ai_score_pass1 = p1.get('review_b_score', 3)
    ai_score_pass2 = p2.get('review_a_score', 3)

    out1 = p1.get('outcome', 'Tie')
    out2 = p2.get('outcome', 'Tie')

    # Position bias check
    if out1 == 'B Win' and out2 == 'A Win':
        final = 'AI Win'
    elif out1 == 'A Win' and out2 == 'B Win':
        final = 'Human Win'
    elif out1 == 'Tie' and out2 == 'Tie':
        final = 'Tie'
    else:
        final = 'Tie (Bias Detected)'
        position_flips += 1

    final_outcomes.append(final)

    # Average over 2 passes
    avg_ai_score = int(round((ai_score_pass1 + ai_score_pass2) / 2))

    if human_score is not None:
        human_scores.append(human_score)
        judge_scores.append(avg_ai_score)

kappa = cohen_kappa_score(human_scores, judge_scores) if len(human_scores) > 0 else None

metrics = {
    'n_samples': len(calibration_set),
    'position_flips_mitigated': position_flips,
    'cohens_kappa': kappa,
    'final_winrate_summary': dict(Counter(final_outcomes))
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nEvaluation Results")
print(json.dumps(metrics, indent=2))


Evaluation Results
{
  "n_samples": 15,
  "position_flips_mitigated": 2,
  "cohens_kappa": 0.0,
  "final_winrate_summary": {
    "Tie": 3,
    "Tie (Bias Detected)": 2
  }
}
